# Part 1 · From a 10x Xenium sample to a simulation-ready region

**VirtualTissue-Xenium** turns a Xenium section into a tissue of AI agents: every cell starts from its
measured state (segmentation, XY position, gene expression → 20 AUCell scores) and, in Parts 2–4, asks
**Jev** which permitted action to take next. This notebook is **Part 1**:

| Part | Notebook | What happens |
|---|---|---|
| **1** | **this one** | Xenium output → QC → provisional cell types → AUCell → candidate regions → **bundle ZIP** |
| 2 | `Part2-4_Run_Jev_Simulation.ipynb` | load and verify the bundle ZIP (yours or from Part 1) |
| 3 | same notebook | choose scenario, duration and API limits; run Jev |
| 4 | same notebook + `Part4_Explore_Saved_Results.ipynb` | interactive replay, figures, tables |

**Example data:** *Xenium Prime Human Lung Cancer FFPE, experiment 2* from
[10x Genomics](https://www.10xgenomics.com/datasets/xenium-human-lung-cancer-post-xenium-technote)
(CC BY 4.0): 278,328 cells, 5,001 genes (Xenium Prime 5K panel). Only 10 files (~503 MiB) of the
22.6 GB archive are needed; step 2 can download them for you.

### Before you start (Colab)
1. **Runtime → Change runtime type → Python 3, CPU.** No GPU. The standard runtime is enough
   (peak memory ≈ 3–4 GB; AUCell on the whole section takes a few minutes).
2. Run the cells **in order**. Every value you may change is a form field marked **✏️**.
3. The four ✏️ places in this notebook: **code source** (step 1, only if not using GitHub),
   **DATA_DIR / OUTPUT_BASE** (step 2), optional **region-size parameters** (step 3), and
   **REGION_TO_EXPORT** (step 8 — this is where you choose `R03`, for example).

**No API key is needed for Part 1.** Save a copy of this notebook in your own Drive to keep your settings.

**Using your own sample?** This workflow targets human tumour tissue with tumour-candidate epithelium,
fibroblasts and immune cells. It requires the five original Xenium files and a reviewed cluster map;
it does not classify cells automatically. Run step 1, then **pause before step 2** and follow the
[own-data guide](https://github.com/hylixinsights/VirtualTissue-Xenium/blob/main/docs/06_your_own_xenium_data.md). It mounts Drive, defines your paths, creates
the data folder, and shows how to edit and validate the cluster map. Copy the four values it prints into
step 2 below. If no automatic proposal meets the tumour-interface criteria, this notebook cannot proceed
to manual region selection.

**Local Jupyter:** input/output form paths do not change automatically. Before step 2, replace `DATA_DIR`
and `OUTPUT_BASE` with local paths (for example `../data/Xenium_Public_Lung5K` and `../outputs/prepared`
when this notebook runs from the repository's `notebooks/` folder). For your own sample, also set the
local cluster-map path and uncheck the public download.

Full guide: [docs/01_part1_prepare_region.md](https://github.com/hylixinsights/VirtualTissue-Xenium/blob/main/docs/01_part1_prepare_region.md) ·
Colab paths and secrets: [docs/05_google_colab.md](https://github.com/hylixinsights/VirtualTissue-Xenium/blob/main/docs/05_google_colab.md)

**Interpretation:** cluster labels are provisional, marker-based interpretations of the 10x clusters, not a
validated classifier. “Tumour-candidate epithelium” does not establish malignancy. Antigen recognition,
protein competence and kinetic rates are not measured by this workflow.


## 1 · Get the code and install dependencies
Installs the exact package versions the engine was validated with (Python 3.12 or 3.13).
If this cell prints **ONE SESSION RESTART IS REQUIRED**, choose **Runtime → Restart session** and run this
cell again (your Drive files are not affected). Continue only when it reports *Ready*.

✏️ Leave `CODE_SOURCE = github` once the repository is public. Before that, upload the whole
`VirtualTissue-Xenium` folder to Google Drive and choose `google_drive`.


In [ ]:
#@title Get the VirtualTissue-Xenium code and install the pinned dependencies
# ✏️ EDIT only if needed — where the code comes from:
#   github        clone the public repository (default)
#   google_drive  copy the VirtualTissue-Xenium folder you placed in Google Drive
#   installed     the package is already installed with pip (local Jupyter)
CODE_SOURCE = "github"  #@param ["github", "google_drive", "installed"]
GITHUB_URL = "https://github.com/hylixinsights/VirtualTissue-Xenium.git"  #@param {type:"string"}
GITHUB_BRANCH = "main"  #@param {type:"string"}
DRIVE_CODE_DIR = "/content/drive/MyDrive/VirtualTissue-Xenium"  #@param {type:"string"}
INSTALL_DEPENDENCIES = True  #@param {type:"boolean"}

import importlib, os, shutil, subprocess, sys
from pathlib import Path
print("Running Python:", sys.version.split()[0], "| executable:", sys.executable)
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
# Temporary Colab disk. Keep the code here, not in Drive.
CODE_DIR = Path("/content/VirtualTissue-Xenium")
if not IN_COLAB and CODE_SOURCE != "installed":
    # Local Jupyter: use the repository that contains this notebook.
    here = Path.cwd().resolve()
    CODE_DIR = here.parent if (here.parent / "src" / "virtualtissue_xenium").is_dir() else here
    CODE_SOURCE = "local_repository"
if CODE_SOURCE == "github":
    if not (CODE_DIR / "src" / "virtualtissue_xenium").is_dir():
        subprocess.check_call(["git", "clone", "--quiet", "--depth", "1", "--branch", GITHUB_BRANCH,
                               GITHUB_URL, str(CODE_DIR)])
elif CODE_SOURCE == "google_drive":
    from google.colab import drive
    drive.mount("/content/drive")
    if not (Path(DRIVE_CODE_DIR) / "src" / "virtualtissue_xenium").is_dir():
        raise FileNotFoundError("DRIVE_CODE_DIR must be the VirtualTissue-Xenium folder "
                                "(the one that contains src/, notebooks/ and README.md).")
    shutil.rmtree(CODE_DIR, ignore_errors=True)
    shutil.copytree(DRIVE_CODE_DIR, CODE_DIR,
                    ignore=shutil.ignore_patterns(".git", "__pycache__", "*.pyc", ".ipynb_checkpoints"))
if CODE_SOURCE != "installed" and str(CODE_DIR / "src") not in sys.path:
    sys.path.insert(0, str(CODE_DIR / "src"))
import virtualtissue_xenium as vtx
from virtualtissue_xenium import colab as vtx_colab
print("VirtualTissue-Xenium", vtx.__version__, "| engine:", vtx.ENGINE_DIR)
os.environ.setdefault("MPLCONFIGDIR", "/tmp/vtx_matplotlib")
os.environ["NUMBA_NUM_THREADS"] = "2"

import colab_setup_prepare
importlib.reload(colab_setup_prepare)
SETUP = colab_setup_prepare.setup_environment(vtx.ENGINE_DIR, install_packages=INSTALL_DEPENDENCIES)
SETUP_READY = SETUP["ready"]

if SETUP_READY:
    prep = vtx.prepare
    from virtualtissue_xenium import download as vtx_download, own_data as vtx_own
    print("Ready. Preparation engine:", prep.VERSION)
else:
    prep = None


## 2 · ✏️ Where the data are and where results go
**DATA_DIR** is the folder that contains `raw/`, `SOURCE.json` and `DOWNLOAD_MANIFEST.json`
(not the `raw/` folder itself). **OUTPUT_BASE** must be outside DATA_DIR; each run creates a new
`run_<timestamp>` subfolder there. Paths starting with `/content/drive/MyDrive/` are in your Google Drive
(Drive is mounted automatically and Colab asks for permission once).

* **Public example:** keep `DOWNLOAD_PUBLIC_EXAMPLE_IF_MISSING` checked. If DATA_DIR is empty, the 10 original
  files (~503 MiB) are downloaded from 10x into it and verified by SHA-256 (a few minutes, once).
* **Your own Xenium run:** complete the
  [own-data guide](https://github.com/hylixinsights/VirtualTissue-Xenium/blob/main/docs/06_your_own_xenium_data.md) first, including mounting Drive before
  accessing its files and validating your edited cluster map. In this form, replace `DATA_DIR`,
  `OUTPUT_BASE` and `CLUSTER_ANNOTATIONS_JSON` with your paths, and **uncheck
  `DOWNLOAD_PUBLIC_EXAMPLE_IF_MISSING`**. The form's assignments replace variables from earlier cells,
  so make these changes here too. Keep the public example and each new sample in separate data folders.


In [ ]:
#@title Data and output folders
if not globals().get("SETUP_READY", False):
    raise RuntimeError("Complete step 1 first. If a restart was requested, select Runtime > Restart session and rerun step 1.")
# ✏️ EDIT: folder that contains raw/, SOURCE.json and DOWNLOAD_MANIFEST.json
DATA_DIR = "/content/drive/MyDrive/VirtualTissue/Xenium_Public_Lung5K"  #@param {type:"string"}
# ✏️ EDIT: where Part 1 writes its results (must be outside DATA_DIR)
OUTPUT_BASE = "/content/drive/MyDrive/VirtualTissue/Xenium_Lung5K_Prepared"  #@param {type:"string"}
DOWNLOAD_PUBLIC_EXAMPLE_IF_MISSING = True  #@param {type:"boolean"}
# ✏️ EDIT only for your own sample: your cluster annotation map (JSON). Empty = public example map.
CLUSTER_ANNOTATIONS_JSON = ""  #@param {type:"string"}

if (DATA_DIR + OUTPUT_BASE + CLUSTER_ANNOTATIONS_JSON).count("/content/drive/") and IN_COLAB:
    vtx_colab.mount_drive()
if not (Path(DATA_DIR) / "DOWNLOAD_MANIFEST.json").is_file():
    if DOWNLOAD_PUBLIC_EXAMPLE_IF_MISSING:
        print("DATA_DIR has no DOWNLOAD_MANIFEST.json: downloading the public example from 10x Genomics...")
        vtx_download.download_public_lung5k(DATA_DIR)
    else:
        raise FileNotFoundError("DOWNLOAD_MANIFEST.json was not found. Set DATA_DIR to the folder containing raw/ and "
                                "DOWNLOAD_MANIFEST.json, not to the raw/ subfolder.")
check = vtx_download.verify_data_folder(DATA_DIR, check_hashes=False)
if not check["ok"]:
    raise ValueError("Input folder problems: " + "; ".join(check["problems"]))
vtx_own.require_annotation_map(DATA_DIR, CLUSTER_ANNOTATIONS_JSON)
print(f"Input: {DATA_DIR} ({check['files']} files present; SHA-256 is checked in step 3)")


## 3 · Region size and AUCell parameters, then load the sample
The defaults reproduce the example: regions of at most **1,000 cells including context**, a target of 750
central cells plus a 30 µm context halo, 8 proposals, AUCell top-100 ranks. The program reduces the central
area when needed to keep the total ≤ 1,000. All cells inside the selected area are kept, including
unclassified and low-RNA cells.

This cell checks every input file against its SHA-256 (set `VERIFY_INPUT_HASHES` off to save ~1 min).


In [ ]:
#@title ✏️ Parameters, integrity check, expression, XY and clusters
from datetime import datetime, timezone
from IPython.display import display, Markdown, clear_output
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

MAX_TOTAL_CELLS = 1000 #@param {type:"integer"}
CORE_TARGET = 750 #@param {type:"integer"}
HALO_UM = 30 #@param {type:"number"}
N_REGIONS = 8 #@param {type:"integer"}
AUCELL_TOP_RANKS = 100 #@param {type:"integer"}
VERIFY_INPUT_HASHES = True #@param {type:"boolean"}
if not 100 <= CORE_TARGET <= MAX_TOTAL_CELLS <= 1000:
    raise ValueError('Use 100 <= CORE_TARGET <= MAX_TOTAL_CELLS <= 1000.')
if not 1 <= N_REGIONS <= 12: raise ValueError('Choose between 1 and 12 region options.')
RUN_DIR=Path(OUTPUT_BASE)/datetime.now(timezone.utc).strftime('run_%Y%m%d_%H%M%S_%f')
if CLUSTER_ANNOTATIONS_JSON:
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    print('Custom cluster map:', vtx_own.use_cluster_annotations(prep, CLUSTER_ANNOTATIONS_JSON, RUN_DIR))
else:
    vtx_own.use_packaged_annotations(prep)
print('Input:',DATA_DIR)
print('Results:',RUN_DIR)
sample=prep.load_sample(DATA_DIR,RUN_DIR,verify_hashes=VERIFY_INPUT_HASHES,min_counts=20,min_genes=10)
prep.save_json(RUN_DIR/'environment.json',prep.environment())
display(pd.DataFrame({'value':[len(sample['obs']),len(sample['genes']),int(sample['obs'].qc_pass.sum())]},
                     index=['Original cells','Gene Expression features','Passed initial RNA QC']))


## 4 · Review provisional cell annotations
The table lists original differential-expression results and markers supporting each interpretation. The plot
shows the fraction of cells with each marker detected. A sparse panel will not detect every marker in every cell.

Mixed groups and cell types without a supported simulation program remain physical context, as do cells with
conflicting lineage markers. Macrophages use a generic provisional class: **SPP1 is absent from this panel**, so
that subtype is not inferred. Review the detailed immune composition: “immune” does not necessarily mean CD8.

✏️ To revise an annotation, edit `ANNOTATION_OVERRIDES` and rerun this cell **and every following cell**.
Example: `{14: {"label":"Unresolved", "compartment":"unknown", "runtime_label":None}}`.


In [ ]:
#@title Editable annotations and marker review
ANNOTATION_OVERRIDES = {}
annotation_table=prep.annotate(sample,ANNOTATION_OVERRIDES)
display(annotation_table[['cluster','n_cells','label','compartment','runtime_label','markers','top_genes','mixed_marker_fraction']])
fig=prep.plot_marker_review(sample)
display(fig);plt.close(fig)


## 5 · Calculate 20 AUCell scores across the entire sample
Uses [decoupler.mt.aucell 2.3.0](https://decoupler.scverse.org/en/latest/api/generated/decoupler.mt.aucell.html)
and 20 human GO:BP definitions from MSigDB 2026.1, on **original counts** (control probes and codewords excluded).
The ranking universe is the whole assayed panel; the default cutoff is **100 ranks (~2% of the panel)**. Cells
with fewer than 100 detected genes keep missing scores, so zero-count genes never enter a scored cell's top ranks.

Outputs: raw AUCell, percentiles within provisional type, and percentiles within type and detection-depth decile
(the latter is what the simulated cells see). **Percentiles across different types do not measure absolute
activity differences**; these are relative RNA-enrichment scores, not demonstrated protein activity.


In [ ]:
#@title Calculate AUCell, reference distributions, and quality checks
from virtualtissue_xenium.own_data import require_tumour_interface
require_tumour_interface(sample)
coverage=prep.score_aucell(sample,n_up=AUCELL_TOP_RANKS,min_overlap=5,batch_size=4096)
display(coverage[['full_genes','panel_genes','panel_coverage','available']])
print('Cells with AUCell:',int(sample['obs'].aucell_available.sum()))
print('Cells without AUCell:',int((~sample['obs'].aucell_available).sum()))
fig,ax=plt.subplots(figsize=(8,3))
ax.hist(sample['obs'].n_genes_by_counts,bins=100,color='#397f9c')
ax.axvline(AUCELL_TOP_RANKS,color='#bc582d',label='Minimum detected genes for AUCell')
ax.set(xlabel='Detected genes per cell',ylabel='Number of cells');ax.legend()
fig.tight_layout();fig.savefig(RUN_DIR/'rna_depth_qc.png',dpi=150)
display(fig);plt.close(fig)


## 6 · Find candidate regions with mixed populations
Each region is a spatial disc, not a hand-picked list of cell types. Automatic proposals require at least 15%
tumour-candidate epithelium, 8% fibroblasts and 15% immune cells in the central area; ranking favours about
35%/20%/30% plus ~10% other stroma. Options overlap little (cell-set Jaccard ≤ 0.30) and are named
**R01, R02, …** in order of balance score. The grey halo is surrounding context and counts toward the
1,000-cell limit. Geometry comes from the original simplified 10x polygons.


In [ ]:
#@title Read segmentation and propose regions
prep.read_geometry(sample)
region_table=prep.propose_regions(sample,n_options=N_REGIONS,core_target=CORE_TARGET,
                                 max_total=MAX_TOTAL_CELLS,halo_um=HALO_UM)
display(region_table[['region_id','n_core','n_halo','n_total','tumor_candidate','fibroblast','immune','stroma','cd8_provisional','macrophage_provisional','supported_qc_core','aucell_fraction','x_um','y_um']])
fig=prep.plot_overview(sample);display(fig);plt.close(fig)


In [ ]:
#@title Region gallery — inspect and compare
fig=prep.plot_regions(sample);display(fig);plt.close(fig)


## 7 · Browse a region (optional)
Use the dropdown to enlarge any option and see its cell-type composition. You can also enter a centre in
micrometres (read it from the whole-section map) and click **Preview manual region**; it becomes the
region `MANUAL`. A manual choice still respects the cell limit and warns if it fails the composition criteria.

This browser only *shows* regions. **The region that is exported is the one you type in step 8.**


In [ ]:
#@title Browse regions
import ipywidgets as widgets
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
selector=widgets.Dropdown(options=[r['region_id'] for r in sample['regions']],value=sample['regions'][0]['region_id'],description='Region:')
preview=widgets.Output()
def show_selected(change=None):
    with preview:
        clear_output(wait=True)
        r=next(r for r in sample['regions'] if r['region_id']==selector.value)
        if not r['balance_pass']:
            display(Markdown('**This manual region does not meet all three minimum composition criteria.**'))
        display(Markdown(f"**{r['region_id']}: {r['n_total']} cells = {r['n_core']} central + {r['n_halo']} surrounding context**"))
        fig=prep.plot_regions(sample,[selector.value],save=False);display(fig);plt.close(fig)
        core_obs=sample['obs'].iloc[r['core']]
        detail=core_obs.groupby(['cell_type_provisional','compartment'],observed=True).agg(
            cells=('internal_id','size'),with_AUCell=('aucell_available','sum'),marker_conflicts=('mixed_marker_flag','sum'))
        display(detail)
selector.observe(show_selected,names='value')
x_input=widgets.FloatText(value=float(region_table.iloc[0].x_um),description='X (µm):')
y_input=widgets.FloatText(value=float(region_table.iloc[0].y_um),description='Y (µm):')
manual_button=widgets.Button(description='Preview manual region',layout=widgets.Layout(width='230px'))
manual_status=widgets.Output()
def choose_manual(_):
    with manual_status:
        clear_output(wait=True)
        try:
            r=prep.manual_region(sample,x_input.value,y_input.value,CORE_TARGET,MAX_TOTAL_CELLS,HALO_UM)
            selector.options=[r['region_id'] for r in sample['regions']]
            selector.value='MANUAL';show_selected()
        except Exception as exc:
            print(str(exc))
manual_button.on_click(choose_manual)
display(widgets.VBox([selector,widgets.HBox([x_input,y_input]),manual_button,manual_status,preview]))
show_selected()


## 8 · ✏️ Choose the region and export the bundle ZIP
**Set `REGION_TO_EXPORT`** to the region you want to simulate — e.g. `R03` — or `MANUAL` after previewing a
manual centre in step 7. Rerun only this cell to export another region (each export gets its own folder).

**TUMOR_MODE** · `exploratory_candidates`: tumour-candidate epithelium acts as `Tumor` (an explicit,
recorded assumption). `context_until_review`: those cells stay as context. No cell is automatically labelled
proliferating tumour.

The export contains `region.h5ad` (raw counts, XY, AUCell), original cell/nucleus boundaries, `atlas.json.gz`,
`runtime_contract.json`, provenance and checksums — see
[docs/02_part2_input_bundle.md](https://github.com/hylixinsights/VirtualTissue-Xenium/blob/main/docs/02_part2_input_bundle.md). A static 10-minute step checks format,
IDs, geometry and numerical initialisation (not biology). **The ZIP is what you give to Part 2.**


In [ ]:
#@title ✏️ Export the chosen region
REGION_TO_EXPORT = "R01"  #@param ["R01", "R02", "R03", "R04", "R05", "R06", "R07", "R08", "MANUAL"] {allow-input: true}
TUMOR_MODE = "exploratory_candidates"  #@param ["exploratory_candidates", "context_until_review"]
from prepared_runtime import validate_bundle
available=[r['region_id'] for r in sample['regions']]
selected_id=REGION_TO_EXPORT.strip().upper()
if selected_id not in available:
    raise ValueError(f'{selected_id} is not available. Choose one of {available} '
                     '(MANUAL exists only after "Preview manual region" in step 7).')
export_stamp=datetime.now(timezone.utc).strftime('%H%M%S_%f')
bundle=prep.export_region(sample,selected_id,tumor_mode=TUMOR_MODE,
                          destination=RUN_DIR/f'simulation_input_{selected_id}_{export_stamp}')
print('Verifying initialization and one static step...')
report=validate_bundle(bundle,smoke_steps=1)
prep.write_bundle_manifest(bundle)
zip_path=prep.zip_bundle(bundle)
display(pd.DataFrame([{k:v for k,v in report.items() if k not in ['fidelity','scope']}]))
print('Prepared folder:',bundle)
print('ZIP bundle:',zip_path)
BUNDLE_ZIP_FOR_PART2=str(zip_path)
display(Markdown(f"**Next:** open `Part2-4_Run_Jev_Simulation.ipynb` and set `BUNDLE_ZIP` to  \n`{zip_path}`  \n"
                 f"(or simply to the folder `{RUN_DIR}` — the newest bundle inside is used)."))


In [ ]:
#@title Optional: also download the ZIP to your computer
DOWNLOAD_ZIP = False #@param {type:"boolean"}
if DOWNLOAD_ZIP and IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
else:
    print('The ZIP is saved at:',zip_path)


## Next step
Open **`Part2-4_Run_Jev_Simulation.ipynb`**. It loads the ZIP, verifies every checksum, shows the measured t0
tissue, estimates the first round of Jev requests (without calling the API), runs the simulation and opens
the interactive replay.

Data: 10x Genomics, experiment 2, [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Gene sets: MSigDB
C5:GO:BP 2026.1.Hs (20 processes). [AUCell method](https://bioconductor.org/packages/release/bioc/vignettes/AUCell/inst/doc/AUCell.html).
